# PolicyAssist AI - Databricks PySpark Insurance Pipeline
## Medallion Architecture (Bronze -> Silver -> Gold Delta Tables)

This notebook demonstrates an end-to-end data engineering pipeline using PySpark and Delta Lake:
- **Bronze Layer**: Raw CSV ingestion with audit metadata.
- **Silver Layer**: Data cleansing, schema enforcement, boundary validation, and standardization.
- **Gold Layer**: Feature engineering (Value for Money index, Affordability & CSR Tiers, Composite Suitability) and analytical aggregations for recommendation models.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit, trim, round as spark_round, when, avg, min as spark_min, max as spark_max, count
from pyspark.sql.types import IntegerType, DoubleType, LongType

# Initialize Spark session
spark = SparkSession.builder \
    .appName("PolicyAssist_DataPipeline") \
    .getOrCreate()

print("Spark Version:", spark.version)

### 1. Bronze Layer (Raw Ingestion)
Reads raw insurance policies CSV file and appends ingestion metadata.

In [ ]:
csv_path = "../data/insurance_policies.csv"
raw_df = spark.read.option("header", "true").option("inferSchema", "false").csv(csv_path)

bronze_df = raw_df \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit("insurance_policies.csv"))

print(f"Bronze Layer Count: {bronze_df.count()}")
bronze_df.show(3, truncate=False)

### 2. Silver Layer (Cleansing & Standardization)
Casts columns to numeric data types, standardizes strings, and enforces data quality checks.

In [ ]:
silver_df = bronze_df \
    .withColumn("policy_id", trim(col("policy_id"))) \
    .withColumn("policy_name", trim(col("policy_name"))) \
    .withColumn("policy_type", trim(col("policy_type"))) \
    .withColumn("company_name", trim(col("company_name"))) \
    .withColumn("min_age", col("min_age").cast(IntegerType())) \
    .withColumn("max_age", col("max_age").cast(IntegerType())) \
    .withColumn("min_income", col("min_income").cast(LongType())) \
    .withColumn("max_family_size", col("max_family_size").cast(IntegerType())) \
    .withColumn("coverage_amount", col("coverage_amount").cast(LongType())) \
    .withColumn("annual_premium", col("annual_premium").cast(DoubleType())) \
    .withColumn("claim_settlement_ratio", col("claim_settlement_ratio").cast(DoubleType())) \
    .withColumn("waiting_period_months", col("waiting_period_months").cast(IntegerType())) \
    .withColumn("cashless_hospitals_count", col("cashless_hospitals_count").cast(IntegerType())) \
    .withColumn("copay_percentage", col("copay_percentage").cast(IntegerType()))

# Filter valid business boundaries
silver_cleaned_df = silver_df.filter(
    (col("annual_premium") > 0) &
    (col("coverage_amount") > 0) &
    (col("claim_settlement_ratio") >= 70.0)
)
print(f"Silver Layer Cleansed Count: {silver_cleaned_df.count()}")
silver_cleaned_df.select("policy_id", "policy_name", "coverage_amount", "annual_premium", "claim_settlement_ratio").show(5)

### 3. Gold Layer (Feature Engineering & Aggregations)
Creates analytical features including value-for-money ratio and policy category summaries.

In [ ]:
gold_features_df = silver_cleaned_df \
    .withColumn("coverage_per_rupee", spark_round(col("coverage_amount") / col("annual_premium"), 2)) \
    .withColumn("affordability_tier", 
        when(col("annual_premium") < 10000, "Budget Friendly")
        .when(col("annual_premium") <= 20000, "Standard")
        .otherwise("Premium")) \
    .withColumn("csr_tier",
        when(col("claim_settlement_ratio") >= 98.0, "Outstanding (>=98%)")
        .when(col("claim_settlement_ratio") >= 95.0, "High (95-98%)")
        .otherwise("Moderate (<95%)"))

gold_summary_df = gold_features_df.groupBy("policy_type").agg(
    count("policy_id").alias("total_policies"),
    spark_round(avg("annual_premium"), 2).alias("avg_premium"),
    spark_round(avg("coverage_amount"), 2).alias("avg_coverage"),
    spark_round(avg("claim_settlement_ratio"), 2).alias("avg_csr")
)

print("Gold Category Aggregation:")
gold_summary_df.show()